In [10]:
%pip -q install uv
!uv pip install --system -q "gradio>=5,<6" plotly scikit-learn pandas numpy openpyxl requests
!uv --version

uv 0.13.9 (x86_64-unknown-linux-gnu)


In [11]:
import os, re, html, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import gradio as gr
from IPython.display import display
warnings.filterwarnings('ignore')

PATRONES = {
    'matricula': 'MEN_MATRICULA_ESTADISTICA_ES_20261119',
    'desercion': 'articles-415244_recurso_18',
    'admitidos': 'articles-425153_recurso',
}
EXTENSIONES = {'matricula': '.csv', 'desercion': '.xlsx', 'admitidos': '.xlsx'}

def encontrar_archivo(tipo):
    prefijo = PATRONES[tipo]
    extension = EXTENSIONES[tipo]
    candidatos = []
    for carpeta in [Path('/content'), Path.cwd(), Path('/mnt/data')]:
        if not carpeta.is_dir():
            continue
        for p in carpeta.glob(prefijo + '*' + extension):
            # Solo nombre exacto o nombre con sufijo numérico entre paréntesis
            sufijo = p.stem[len(prefijo):]
            if sufijo == '' or re.fullmatch(r'\(\d+\)', sufijo):
                if p.is_file():
                    candidatos.append(p)
    if not candidatos:
        return None

    candidatos = list({str(p.resolve()): p for p in candidatos}.values())
    return max(candidatos, key=lambda p: p.stat().st_mtime)

RUTAS = {tipo: encontrar_archivo(tipo) for tipo in PATRONES}
faltantes = [tipo for tipo, ruta in RUTAS.items() if ruta is None]
if faltantes:
    try:
        from google.colab import files
        print('Sube los archivos faltantes:', ', '.join(faltantes))
        files.upload()
    except ImportError:
        pass
    RUTAS = {tipo: encontrar_archivo(tipo) for tipo in PATRONES}

faltantes = [tipo for tipo, ruta in RUTAS.items() if ruta is None]
if faltantes:
    raise FileNotFoundError('No se encontraron: ' + ', '.join(faltantes) +
                            '. Comprueba que los archivos estén en /content.')
print('Archivos encontrados:')
for tipo, ruta in RUTAS.items():
    print(f'  {tipo}: {ruta.name}')

Archivos encontrados:
  matricula: MEN_MATRICULA_ESTADISTICA_ES_20261119.csv
  desercion: articles-415244_recurso_18.xlsx
  admitidos: articles-425153_recurso.xlsx


In [12]:
#Columnas dentro del CSV de matriculas para utilizar (no todas las columnas son utilizadas dada la cantidad de datos)
cols=['Código de la Institución','IES PADRE','Institución de Educación Superior (IES)',
 'Id_Sector','Departamento de domicilio de la IES','Código SNIES delprograma',
 'Programa Académico','Id_Nivel_Formacion','Año','Semestre','Total Matriculados']

m=pd.read_csv(RUTAS['matricula'],usecols=cols,low_memory=False,encoding='utf-8-sig')

m=m.rename(columns={'Código de la Institución':'codigo_ies','IES PADRE':'codigo_padre',
 'Institución de Educación Superior (IES)':'institucion','Id_Sector':'sector_id',
 'Departamento de domicilio de la IES':'departamento','Código SNIES delprograma':'codigo_programa',
 'Programa Académico':'programa','Id_Nivel_Formacion':'nivel_id','Año':'anio',
 'Semestre':'semestre','Total Matriculados':'matriculados'})

for c in ['codigo_ies','codigo_padre','codigo_programa','anio','semestre','matriculados','nivel_id','sector_id']:

 m[c]=pd.to_numeric(m[c],errors='coerce')

m=m.dropna(subset=['codigo_padre','anio','matriculados'])

m=m[m.matriculados>=0].copy()

m[['codigo_padre','anio']]=m[['codigo_padre','anio']].astype(int)

m['sector']=m.sector_id.map({1:'Oficial',2:'Privada'}).fillna('Sin dato')

#Dentro del documento, TyT se usa como abreviatura para Tecnico y Tecnologo en cuanto a formacion academica.
m['nivel_spadies']=m.nivel_id.map({1:'TyT',2:'TyT',3:'Universitario',4:'Especialización',5:'Maestría',6:'Universitario',7:'Especialización',8:'Maestría',9:'Doctorado'}).fillna('Sin homologar')

m['departamento']=m.departamento.fillna('Sin dato').astype(str).str.strip()
m['programa']=m.programa.fillna('Sin dato').astype(str).str.strip()
m['institucion']=m.institucion.fillna('Sin dato').astype(str).str.strip()

keys=['codigo_padre','anio','departamento','sector']
sem=m.groupby(keys+['semestre'],as_index=False)['matriculados'].sum()
mat=sem.groupby(keys,as_index=False)['matriculados'].mean()
mat['matriculados']=mat.matriculados.round().astype(int)
print('Filas CSV:',len(m),'| IES-año:',len(mat),'| años:',m.anio.min(),m.anio.max())

Filas CSV: 390903 | IES-año: 2114 | años: 2015 2021


In [13]:
#Columnas dentro del XLSX para las estadisticas de deserción oficiales del SPADIES

s=pd.read_excel(RUTAS['desercion'],sheet_name='TDA IES Padre Total',header=11)

s=s.loc[:,~s.columns.astype(str).str.startswith('Unnamed')]

s=s.rename(columns={'CODIGO IES PADRE':'codigo_padre','IES':'ies_spadies'})

years=[c for c in s.columns if str(c).isdigit() and 2010<=int(c)<=2100]

s=s.melt(id_vars=['codigo_padre','ies_spadies'],value_vars=years,var_name='anio',value_name='tda')

for c in ['codigo_padre','anio','tda']:s[c]=pd.to_numeric(s[c],errors='coerce')

s=s.dropna(subset=['codigo_padre','anio','tda'])

s=s[s.tda.between(0,1)].copy()

s[['codigo_padre','anio']]=s[['codigo_padre','anio']].astype(int)

s=s.drop_duplicates(['codigo_padre','anio'])

print('TDA oficial:',len(s),'filas; años',s.anio.min(),'-',s.anio.max())

#Columnas de adminitos dentro de la plantilla de excel
ad=pd.read_excel(RUTAS['admitidos'],sheet_name='1.',header=5)
print('Columnas admitidos relevantes:',[c for c in ad.columns if 'ADMIT' in str(c).upper() or 'TOTAL' in str(c).upper()][:15])
display(ad.head(2))


ad['ADMITIDOS']=pd.to_numeric(ad['ADMITIDOS'],errors='coerce')
ad=ad.dropna(subset=['ADMITIDOS'])
ad=ad[ad['ADMITIDOS']>=0]
TOTAL_ADMITIDOS_2023=int(ad['ADMITIDOS'].sum())
print('Admitidos reportados 2023:',f'{TOTAL_ADMITIDOS_2023:,}')

TDA oficial: 3982 filas; años 2010 - 2024
Columnas admitidos relevantes: ['ADMITIDOS']


,CÓDIGO DE LA INSTITUCIÓN,IES PADRE,INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES),TIPO IES,ID SECTOR IES,SECTOR IES,ID CARÁCTER IES,CARÁCTER IES,CÓDIGO DEL DEPARTAMENTO (IES),DEPARTAMENTO DE DOMICILIO DE LA IES,...,DESC CINE CAMPO DETALLADO,CÓDIGO DEL DEPARTAMENTO (PROGRAMA),DEPARTAMENTO DE OFERTA DEL PROGRAMA,CÓDIGO DEL MUNICIPIO (PROGRAMA),MUNICIPIO DE OFERTA DEL PROGRAMA,ID SEXO,SEXO,AÑO,SEMESTRE,ADMITIDOS
0,1101,1101.0,UNIVERSIDAD NACIONAL DE COLOMBIA,Principal,1.0,Oficial,4.0,Universidad,11.0,"Bogotá, D.C.",...,Producción agrícola y ganadera,11.0,"Bogotá, D.C.",11001.0,"Bogotá, D.C.",1.0,Masculino,2023.0,1.0,67.0
1,1101,1101.0,UNIVERSIDAD NACIONAL DE COLOMBIA,Principal,1.0,Oficial,4.0,Universidad,11.0,"Bogotá, D.C.",...,Producción agrícola y ganadera,11.0,"Bogotá, D.C.",11001.0,"Bogotá, D.C.",1.0,Masculino,2023.0,2.0,65.0


Admitidos reportados 2023: 1,243,778


In [14]:
#Documentacion de columnas del SNIES
panel=mat.merge(s[['codigo_padre','anio','tda','ies_spadies']],on=['codigo_padre','anio'],how='left',validate='many_to_one')

print('IES-año con matrícula:',len(panel),'| con TDA oficial:',panel.tda.notna().sum())
print('Cobertura:',f'{panel.tda.notna().mean():.1%}')


programas=m.groupby(['anio','codigo_padre','codigo_programa','programa','departamento','sector'],dropna=False,as_index=False)['matriculados'].sum()

print('Columnas de admitidos:')
print(list(ad.columns))
display(panel.head())

IES-año con matrícula: 2114 | con TDA oficial: 2048
Cobertura: 96.9%
Columnas de admitidos:
['CÓDIGO DE LA INSTITUCIÓN', 'IES PADRE', 'INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)', 'TIPO IES', 'ID SECTOR IES', 'SECTOR IES', 'ID CARÁCTER IES', 'CARÁCTER IES', 'CÓDIGO DEL DEPARTAMENTO (IES)', 'DEPARTAMENTO DE DOMICILIO DE LA IES', 'CÓDIGO DEL MUNICIPIO IES', 'MUNICIPIO DE DOMICILIO DE LA IES', 'IES ACREDITADA', 'CÓDIGO SNIES DEL PROGRAMA', 'PROGRAMA ACADÉMICO', 'PROGRAMA ACREDITADO', 'ID NIVEL ACADÉMICO', 'NIVEL ACADÉMICO', 'ID NIVEL DE FORMACIÓN', 'NIVEL DE FORMACIÓN', 'ID MODALIDAD', 'MODALIDAD', 'ID ÁREA', 'ÁREA DE CONOCIMIENTO', 'ID NÚCLEO', 'NÚCLEO BÁSICO DEL CONOCIMIENTO (NBC)', 'ID CINE CAMPO AMPLIO', 'DESC CINE CAMPO AMPLIO', 'ID CINE CAMPO ESPECIFICO', 'DESC CINE CAMPO ESPECIFICO', 'ID CINE CAMPO DETALLADO', 'DESC CINE CAMPO DETALLADO', 'CÓDIGO DEL DEPARTAMENTO (PROGRAMA)', 'DEPARTAMENTO DE OFERTA DEL PROGRAMA', 'CÓDIGO DEL MUNICIPIO (PROGRAMA)', 'MUNICIPIO DE OFERTA DEL PROGRAMA', 

,codigo_padre,anio,departamento,sector,matriculados,tda,ies_spadies
0,1101,2015,AMAZONAS,Oficial,72,0.049815,UNIVERSIDAD NACIONAL DE COLOMBIA
1,1101,2015,ANTIOQUIA,Oficial,24822,0.049815,UNIVERSIDAD NACIONAL DE COLOMBIA
2,1101,2015,BOGOTA D.C.,Oficial,62750,0.049815,UNIVERSIDAD NACIONAL DE COLOMBIA
3,1101,2015,CALDAS,Oficial,11236,0.049815,UNIVERSIDAD NACIONAL DE COLOMBIA
4,1101,2015,SAN ANDRES Y PROVIDENCIA,Oficial,56,0.049815,UNIVERSIDAD NACIONAL DE COLOMBIA


In [15]:

# Modelo temporal usando únicamente SPADIES.
hist=s[['codigo_padre','ies_spadies','anio','tda']].sort_values(['codigo_padre','anio']).copy()
hist['tda_previa']=hist.groupby('codigo_padre')['tda'].shift(1)
hist['anio_previo']=hist.groupby('codigo_padre')['anio'].shift(1)
hist.loc[hist.anio-hist.anio_previo!=1,'tda_previa']=np.nan
entrenable=hist.dropna(subset=['tda','tda_previa']).copy()
features_hist=['tda_previa','anio']
modelo_hist=None; mae_hist=None; anio_test_hist=None
if len(entrenable)>=100:
    anio_test_hist=int(entrenable.anio.max())  # 2024, si hay registros suficientes
    train=entrenable[entrenable.anio<anio_test_hist]
    test=entrenable[entrenable.anio==anio_test_hist]
    if len(train)>=30 and len(test)>=10:
        evaluador=RandomForestRegressor(n_estimators=180,min_samples_leaf=4,random_state=42,n_jobs=-1)
        evaluador.fit(train[features_hist],train.tda)
        mae_hist=mean_absolute_error(test.tda,evaluador.predict(test[features_hist]))
    modelo_hist=RandomForestRegressor(n_estimators=180,min_samples_leaf=4,random_state=42,n_jobs=-1)
    modelo_hist.fit(entrenable[features_hist],entrenable.tda)

def pronostico_ies(d, anio):
    """Pronóstico para año+1 de las IES que tienen tasa observada en anio."""
    if modelo_hist is None or d.empty:
        return None
    d=d.dropna(subset=['tda']).copy()
    if d.empty: return None
    X=pd.DataFrame({'tda_previa':d.tda.to_numpy(), 'anio':np.full(len(d),anio+1)})
    return float(np.clip(modelo_hist.predict(X),0,1).mean())

print('Modelo SPADIES:', 'disponible' if modelo_hist else 'no disponible',
      '| año de validación:',anio_test_hist,
      '| MAE:',round(mae_hist*100,2) if mae_hist is not None else None,'puntos porcentuales')
print('Cobertura SPADIES:',int(s.anio.min()),'-',int(s.anio.max()))
print('Cobertura SNIES:',int(m.anio.min()),'-',int(m.anio.max()))


Modelo SPADIES: disponible | año de validación: 2024 | MAE: 4.21 puntos porcentuales
Cobertura SPADIES: 2010 - 2024
Cobertura SNIES: 2015 - 2021


In [17]:

#Codificación de Gradio para la interfaz interactiva de usuario (Optimizada con Inteligencia Artificial dada la escases de tiempo para estudiar la codificación)


CSS = '''
.gradio-container {
    max-width: 1300px !important;
    background: #edf2f6 !important;
    font-family: system-ui, sans-serif;
}
.hero {
    background: #132a4a;
    border-radius: 16px;
    padding: 25px;
    color: white;
}
.hero h1 {
    color: white !important;
    font-family: Georgia, serif;
    font-size: 29px;
    margin: 8px 0;
}
.hero p { color: #dce5ef; }
.kpis {
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 12px;
}
.kpi {
    background: white;
    border: 1px solid #dbe3ec;
    border-radius: 12px;
    padding: 16px;
}
.kpi small { color: #65758c; }
.kpi strong {
    display: block;
    color: #1d4e7a;
    font-size: 29px;
    margin: 9px 0;
}
.hint { color: #65758c; font-size: 12px; }
.section {
    font-size: 18px;
    font-weight: 700;
    color: #243d58;
    margin: 8px 0;
}
.tablewrap { overflow-x: auto; }
table.ed {
    border-collapse: collapse;
    width: 100%;
    background: white;
    font-size: 12px;
}
table.ed td, table.ed th {
    padding: 9px;
    border-bottom: 1px solid #e5e9ef;
    text-align: left;
}
table.ed th {
    background: #edf2f7;
    color: #43556b;
}
@media(max-width:800px) {
    .kpis { grid-template-columns: repeat(2, 1fr); }
}
'''

def fmt_n(x):
    return f'{x:,.0f}'.replace(',', '.')

def fmt_p(x):
    return 'Sin dato' if pd.isna(x) else f'{100*x:.2f}'.replace('.', ',') + ' %'

def media_ponderada(d):
    v = d.dropna(subset=['tda'])
    return np.average(
        v.tda,
        weights=v.matriculados.clip(lower=1)
    ) if len(v) else np.nan

def filtrar(d, dep, sec):
    if dep != 'Todos':
        d = d[d.departamento == dep]
    if sec != 'Todos':
        d = d[d.sector == sec]
    return d


# ----------------------------------------------------
# VISTA INTEGRADA: SNIES + SPADIES
# ----------------------------------------------------

def render_integrada(anio, departamento, sector, programa):
    anio = int(anio)
    d = filtrar(panel, departamento, sector)
    da = d[d.anio == anio].copy()

    prog = programas[programas.anio == anio].copy()

    if departamento != 'Todos':
        prog = prog[prog.departamento == departamento]

    if sector != 'Todos':
        prog = prog[prog.sector == sector]

    if programa != 'Todos los programas':
        codigos = prog.loc[
            prog.programa == programa, 'codigo_padre'
        ].unique()

        prog = prog[prog.programa == programa]
        da = da[da.codigo_padre.isin(codigos)]
        d = d[d.codigo_padre.isin(codigos)]

    # Matrícula filtrada a programa cuando corresponde.
    # No se atribuyen tasas institucionales a programas.
    if programa != 'Todos los programas':
        matricula = (
            prog.matriculados.sum() /
            max(1, m[m.anio == anio].semestre.nunique())
        )
    else:
        matricula = da.matriculados.sum()

    tasa = media_ponderada(da)
    cub = da.tda.notna().sum()
    total = len(da)
    pred = pronostico_ies(da, anio)

    kpis = [
        (
            'Matrícula (promedio semestral)',
            fmt_n(matricula),
            'SNIES · registros del filtro'
        ),
        (
            'TDA institucional (ponderada)',
            fmt_p(tasa),
            'Promedio calculado, no tasa oficial territorial'
        ),
        (
            'IES con tasa disponible',
            f'{cub} / {total}',
            'Cruce matrícula–SPADIES'
        ),
        (
            'Pronóstico TDA siguiente año',
            fmt_p(pred),
            'Modelo experimental; no individual'
        )
    ]

    tarjetas = (
        '<div class="kpis">' +
        ''.join(
            f'<div class="kpi">'
            f'<small>{html.escape(a)}</small>'
            f'<strong>{html.escape(str(b))}</strong>'
            f'<div class="hint">{html.escape(c)}</div>'
            f'</div>'
            for a, b, c in kpis
        ) +
        '</div>'
    )

    # Gráfica 1: evolución de la deserción
    v = (
        d.dropna(subset=['tda'])
        .groupby('anio')
        .apply(media_ponderada, include_groups=False)
        .sort_index()
    )

    f1 = go.Figure()

    if len(v):
        f1.add_scatter(
            x=v.index,
            y=v.values * 100,
            mode='lines+markers',
            line_color='#2563a0',
            fill='tozeroy',
            name='Media ponderada'
        )

    f1.update_layout(
        title='Evolución de TDA institucional (promedio ponderado)',
        template='plotly_white',
        height=330,
        yaxis_title='%',
        margin=dict(l=30, r=15, t=55, b=25)
    )

    # Gráfica 2: comparación territorial
    v2 = filtrar(panel, 'Todos', sector)
    v2 = v2[v2.anio == anio]

    if programa != 'Todos los programas':
        v2 = v2[v2.codigo_padre.isin(prog.codigo_padre.unique())]

    if departamento != 'Todos':
        v2 = v2[v2.departamento == departamento]

    z = (
        v2.dropna(subset=['tda'])
        .groupby('departamento')
        .apply(media_ponderada, include_groups=False)
        .dropna()
        .sort_values(ascending=False)
        .head(10)
        .sort_values()
    )

    f2 = go.Figure(
        go.Bar(
            x=z.values * 100,
            y=z.index,
            orientation='h',
            marker_color='#447aa5'
        )
    )

    f2.update_layout(
        title='Comparación territorial (proxy, NO TDA oficial)',
        template='plotly_white',
        height=330,
        xaxis_title='%',
        margin=dict(l=10, r=20, t=55, b=30)
    )

    # Tabla de programas con mayor matrícula
    top = (
        prog.groupby(
            ['programa', 'codigo_padre'],
            as_index=False
        )
        .matriculados.sum()
        .sort_values('matriculados', ascending=False)
        .head(10)
    )

    nombres = (
        panel[['codigo_padre', 'ies_spadies']]
        .dropna()
        .drop_duplicates('codigo_padre')
    )

    top = top.merge(
        nombres,
        on='codigo_padre',
        how='left'
    )

    rows = ''.join(
        '<tr><td>' +
        html.escape(str(r.programa)) +
        '</td><td>' +
        html.escape(
            str(r.ies_spadies)
            if pd.notna(r.ies_spadies)
            else str(r.codigo_padre)
        ) +
        '</td><td>' +
        fmt_n(r.matriculados) +
        '</td></tr>'
        for r in top.itertuples()
    )

    tabla = (
        '<div class="tablewrap"><table class="ed">'
        '<tr><th>Programa</th><th>IES padre</th>'
        '<th>Matrícula sumada por semestres*</th></tr>' +
        rows +
        '</table></div>'
        '<div class="hint">'
        '*Tabla de matrícula, NO deserción por programa. '
        'La suma incluye semestres y no representa '
        'personas únicas del año.'
        '</div>'
    )

    info = (
        f'<div class="section">Predicción experimental {anio+1}</div>'
        f'<h2>{fmt_p(pred)}</h2>'
        +
        (
            '<p>Pronóstico agregado de TDA de las IES seleccionadas, '
            'ponderado por matrícula. No representa probabilidad '
            'de abandono de un estudiante.</p>'
            if pred is not None
            else '<p>No hay suficientes datos observados para pronosticar.</p>'
        )
        +
        (
            f'<p>Validación temporal ({anio_test_hist}): '
            f'MAE {mae_hist*100:.2f} puntos porcentuales.</p>'
            if mae_hist is not None
            else '<p>Modelo sin validación suficiente.</p>'
        )
        +
        '<p class="hint">Entrenado con TDA histórica y '
        'matrícula previa; interpretación exploratoria.</p>'
    )

    return tarjetas, f1, f2, tabla, info


# ----------------------------------------------------
# VISTA HISTÓRICA: SPADIES 2010–2024
# ----------------------------------------------------

def render_historica(anio):
    anio = int(anio)

    actual = (
        s[s.anio == anio]
        .dropna(subset=['tda'])
        .copy()
    )

    # Media simple entre instituciones reportantes.
    # No es la tasa nacional oficial.
    media = actual.tda.mean() if len(actual) else np.nan
    pred = pronostico_ies(actual, anio)

    cards = [
        (
            'IES con TDA reportada',
            fmt_n(len(actual)),
            'SPADIES · año seleccionado'
        ),
        (
            'Media de TDA entre IES',
            fmt_p(media),
            'Media simple; NO tasa nacional oficial'
        ),
        (
            'Cobertura histórica',
            '2010–2024',
            'SPADIES · TDA institucional'
        ),
        (
            'Pronóstico IES siguiente año',
            fmt_p(pred),
            'Experimental · media de pronósticos'
        )
    ]

    tarjetas = (
        '<div class="kpis">' +
        ''.join(
            f'<div class="kpi">'
            f'<small>{html.escape(a)}</small>'
            f'<strong>{html.escape(str(b))}</strong>'
            f'<div class="hint">{html.escape(c)}</div>'
            f'</div>'
            for a, b, c in cards
        ) +
        '</div>'
    )

    # Gráfica 1: evolución histórica
    tendencia = (
        s.dropna(subset=['tda'])
        .groupby('anio')['tda']
        .mean()
        .sort_index()
    )

    f1 = go.Figure()

    f1.add_scatter(
        x=tendencia.index,
        y=tendencia.values * 100,
        mode='lines+markers',
        line_color='#2563a0',
        name='Media de TDA entre IES'
    )

    f1.add_vline(
        x=2021,
        line_dash='dot',
        line_color='#888'
    )

    f1.update_layout(
        title='Evolución de la media institucional · SPADIES 2010–2024',
        template='plotly_white',
        height=330,
        yaxis_title='%',
        margin=dict(l=35, r=15, t=55, b=25)
    )

    # Gráfica 2: IES con mayor TDA
    top = (
        actual.nlargest(10, 'tda')
        .sort_values('tda')
    )

    f2 = go.Figure(
        go.Bar(
            x=top.tda * 100,
            y=top.ies_spadies,
            orientation='h',
            marker_color='#447aa5'
        )
    )

    f2.update_layout(
        title='10 IES con mayor TDA reportada · ' + str(anio),
        template='plotly_white',
        height=330,
        xaxis_title='%',
        margin=dict(l=10, r=20, t=55, b=30)
    )

    # Tabla de instituciones
    vista = (
        actual.sort_values('tda', ascending=False)
        .head(12)
    )

    rows = ''.join(
        '<tr><td>' +
        html.escape(str(r.ies_spadies)) +
        '</td><td>' +
        fmt_p(r.tda) +
        '</td></tr>'
        for r in vista.itertuples()
    )

    tabla = (
        '<div class="tablewrap"><table class="ed">'
        '<tr><th>Institución (IES padre)</th>'
        '<th>TDA oficial</th></tr>' +
        rows +
        '</table></div>'
        '<p class="hint">'
        'TDA institucional oficial, no tasa por programa.'
        '</p>'
    )

    nota = (
        f'<div class="section">Predicción experimental para {anio+1}</div>'
        f'<h2>{fmt_p(pred)}</h2>'
        '<p>Media simple de pronósticos de TDA institucional; '
        'no es tasa nacional ni riesgo individual.</p>'
        +
        (
            f'<p>Evaluación temporal en {anio_test_hist}: '
            f'MAE {mae_hist*100:.2f} puntos porcentuales.</p>'
            if mae_hist is not None
            else '<p>Sin evaluación temporal suficiente.</p>'
        )
        +
        '<p class="hint">Modelo entrenado con la TDA del año '
        'anterior y el año. Para pronosticar años futuros se '
        'necesita la TDA del año inmediatamente anterior.</p>'
    )

    return tarjetas, f1, f2, tabla, nota


# ----------------------------------------------------
# TABLA 1: DESERCIÓN ANUAL 2010–2024
# ----------------------------------------------------

# Promedio y mediana calculados con las tasas
# institucionales oficiales de SPADIES.
# NO equivalen a la tasa nacional oficial.

resumen_anual = (
    s.dropna(subset=['tda'])
    .groupby('anio', as_index=False)
    .agg(
        ies_reportantes=('codigo_padre', 'nunique'),
        promedio_tda=('tda', 'mean'),
        mediana_tda=('tda', 'median')
    )
    .sort_values('anio', ascending=False)
)

resumen_anual['promedio_tda'] = (
    resumen_anual['promedio_tda'] * 100
).round(2)

resumen_anual['mediana_tda'] = (
    resumen_anual['mediana_tda'] * 100
).round(2)

resumen_anual = resumen_anual.rename(columns={
    'anio': 'Año',
    'ies_reportantes': 'IES reportantes',
    'promedio_tda': 'Promedio TDA (%)',
    'mediana_tda': 'Mediana TDA (%)'
})


# ----------------------------------------------------
# TABLA 2: DESERCIÓN POR INSTITUCIÓN
# ----------------------------------------------------

def tabla_ies_anio(anio):
    d = (
        s[s.anio == int(anio)]
        [['codigo_padre', 'ies_spadies', 'tda']]
        .copy()
    )

    d['tda'] = (d['tda'] * 100).round(2)

    return (
        d.rename(columns={
            'codigo_padre': 'Código IES padre',
            'ies_spadies': 'Institución',
            'tda': 'TDA oficial (%)'
        })
        .sort_values('TDA oficial (%)', ascending=False)
    )


# ----------------------------------------------------
# ACTUALIZAR DASHBOARD SEGÚN FILTROS
# ----------------------------------------------------

def actualizar_vistas(vista, anio, departamento, sector, programa):

    if vista == 'Histórica SPADIES (2010–2024)':
        return (
            *render_historica(anio),
            tabla_ies_anio(anio)
        )

    if int(anio) > int(panel.anio.max()):
        vacio = go.Figure().update_layout(
            template='plotly_white',
            height=330
        )

        msg = (
            '<p>No hay matrícula SNIES para este año en el CSV. '
            'Cambia a «Histórica SPADIES» para consultar '
            'la TDA hasta 2024.</p>'
        )

        return (
            msg,
            vacio,
            vacio,
            msg,
            msg,
            tabla_ies_anio(anio)
        )

    return (
        *render_integrada(anio, departamento, sector, programa),
        tabla_ies_anio(anio)
    )


# ----------------------------------------------------
# CREACIÓN DE LA INTERFAZ GRADIO
# ----------------------------------------------------

ANIOS_TODOS = sorted(
    s.anio.dropna().astype(int).unique().tolist(),
    reverse=True
)

departamentos = (
    ['Todos'] +
    sorted(panel.departamento.dropna().unique().tolist())
)

sectores = (
    ['Todos'] +
    sorted(panel.sector.dropna().unique().tolist())
)

programas_op = (
    ['Todos los programas'] +
    sorted(programas.programa.dropna().unique().tolist())
)

with gr.Blocks(
    css=CSS,
    title='EduData Colombia · ETITC · 2024'
) as demo:

    gr.HTML(
        '<div class="hero">'
        '<small>ETITC · CIENCIA DE DATOS</small>'
        '<h1>Sistema de Análisis y Predicción de '
        'Deserción Estudiantil en Educación Superior Colombiana</h1>'
        '<p>SPADIES 2010–2024 · '
        'SNIES matrícula 2015–2021 · '
        'SNIES admitidos 2023</p>'
        '</div>'
    )

    gr.HTML(
        '<div class="section">'
        'Vista general · selecciona la fuente y los filtros'
        '</div>'
    )

    vista = gr.Radio(
        [
            'Histórica SPADIES (2010–2024)',
            'Integrada SNIES + SPADIES (2015–2021)'
        ],
        value='Histórica SPADIES (2010–2024)',
        label='Modo de consulta'
    )

    with gr.Row():
        ya = gr.Dropdown(
            ANIOS_TODOS,
            value=2024,
            label='Año'
        )

        de = gr.Dropdown(
            departamentos,
            value='Todos',
            label='Departamento (solo vista integrada)'
        )

        se = gr.Dropdown(
            sectores,
            value='Todos',
            label='Sector (solo vista integrada)'
        )

        pr = gr.Dropdown(
            programas_op,
            value='Todos los programas',
            label='Programa (solo vista integrada)'
        )

    gr.HTML(
        f'<p class="hint">'
        'En la vista histórica, los filtros de departamento, '
        'sector y programa no se aplican: SPADIES proporciona '
        'TDA por IES padre. Admitidos SNIES 2023: '
        f'<b>{fmt_n(TOTAL_ADMITIDOS_2023)}</b> '
        '(dato complementario independiente).'
        '</p>'
    )

    # Indicadores principales
    gr.HTML(
        '<div class="section">Indicadores principales</div>'
    )

    k = gr.HTML()

    # Gráficas
    with gr.Row():
        g1 = gr.Plot(show_label=False)
        g2 = gr.Plot(show_label=False)

    # Detalle y predicción
    with gr.Row():
        with gr.Column(scale=3):
            gr.HTML(
                '<div class="section">'
                'Detalle institucional o de matrícula'
                '</div>'
            )
            tb = gr.HTML()

        with gr.Column(scale=2):
            pdv = gr.HTML()

    # TABLA 1: DESERCIÓN ANUAL
    gr.HTML(
        '<div class="section">'
        'Tabla de deserción anual · SPADIES 2010–2024'
        '</div>'
    )

    gr.Markdown(
        '**Promedio y mediana calculados entre las IES '
        'reportantes**. No representan la tasa nacional '
        'oficial de deserción.'
    )

    gr.Dataframe(
        value=resumen_anual,
        interactive=False,
        wrap=True,
        label='Resumen anual de TDA',
        max_height=360
    )

    # TABLA 2: DESERCIÓN POR INSTITUCIÓN
    gr.HTML(
        '<div class="section">'
        'Deserción por institución · año seleccionado'
        '</div>'
    )

    tabla_instituciones = gr.Dataframe(
        interactive=False,
        wrap=True,
        label='TDA oficial por IES padre',
        max_height=380
    )

    gr.HTML(
        '<p class="hint">'
        'Fuente: MEN / SPADIES y SNIES. '
        'La TDA de cada IES es oficial; la media entre IES '
        'y los pronósticos son cálculos propios. '
        'No hay matrícula 2022–2024 en el CSV proporcionado. '
        'No se calculan desertores individuales.'
        '</p>'
    )

    # Entradas y salidas
    inputs = [vista, ya, de, se, pr]

    outputs = [
        k,
        g1,
        g2,
        tb,
        pdv,
        tabla_instituciones
    ]

    # Actualizar al cambiar filtros
    for inp in inputs:
        inp.change(
            fn=actualizar_vistas,
            inputs=inputs,
            outputs=outputs
        )

    # Cargar la vista inicial
    demo.load(
        fn=actualizar_vistas,
        inputs=inputs,
        outputs=outputs
    )

# Ejecutar el dashboard
demo.launch(share=True)



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://76c1ef985c24e48bdf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Cómo presentar los resultados

1. **Vista histórica SPADIES (2010–2024):** tasas oficiales por IES padre, media simple entre instituciones y pronóstico experimental de TDA para el año siguiente. **La media simple no es la tasa nacional oficial**.
2. **Vista integrada SNIES + SPADIES (2015–2021):** matrícula y filtros de departamento, sector y programa, con tasas institucionales cuando existe cruce.
3. **Admitidos 2023:** cifra independiente; no es matrícula ni deserción.
4. **Predicción:** Random Forest con TDA previa, validado dejando el último año como prueba. Error MAE expresado en puntos porcentuales. No predice deserción individual.
5. **Límite:** el CSV suministrado no incluye matrícula 2022–2024. No se inventan cifras faltantes.

6. **Tablas Gradio:** resumen anual SPADIES 2010–2024 (media y mediana de TDA institucional, no tasa nacional oficial) y detalle de todas las IES del año seleccionado.
